In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("debug-bronze-read")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "admin123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

In [25]:
orders_df_bronze = spark.read.parquet("s3a://bronze/orders/")

In [26]:
orders_df_bronze.limit(1).toPandas()

,kafka_key,raw_json,topic,partition,offset,kafka_timestamp,bronze_ingestion_time
0,489436,"{""event_type"": ""order_item_created"", ""invoice_...",retail_order_events,0,0,2026-04-07 12:23:02.478,2026-04-07 12:26:00.403


In [27]:
from pyspark.sql.functions import get_json_object

orders_df_bronze.filter(
    get_json_object(col("raw_json"), "$.event_type") == "order_item_cancelled"
).count()

10499

In [22]:
orders_df = spark.read.parquet("s3a://silver/orders/")

In [23]:
orders_df.count()

512093

In [24]:
orders_df.filter(col('event_type') == 'order_item_cancelled').count()

12218

In [12]:
from pyspark.sql.functions import *
cust_ids_nan_df = orders_df.filter(
    col("customer_id").isNull() | (col("customer_id") == "")
)
cust_ids_nan_df.count()

107380

In [20]:
customers_df = spark.read.parquet("s3a://silver/customers/")

In [21]:
customers_df.limit(10).toPandas()

,kafka_key,topic,partition,offset,kafka_timestamp,bronze_ingestion_time,event_type,customer_id,country,first_seen_time,silver_ingestion_time
0,12346.0,retail_customer_events,1,390,2026-04-07 12:42:57.775,2026-04-07 12:42:57.790,customer_activity,12346,United Kingdom,2009-12-14 08:34:00,2026-04-22 05:47:37.601075
1,12347.0,retail_customer_events,0,2035,2026-04-08 06:48:44.835,2026-04-08 06:48:44.852,customer_activity,12347,Iceland,2010-10-31 14:20:00,2026-04-22 05:47:37.601075
2,12348.0,retail_customer_events,0,1825,2026-04-08 06:02:32.444,2026-04-08 06:02:32.459,customer_activity,12348,Finland,2010-09-27 14:59:00,2026-04-22 05:47:37.601075
3,12349.0,retail_customer_events,1,162,2026-04-07 12:29:57.256,2026-04-07 12:29:57.307,customer_activity,12349,Italy,2009-12-04 12:49:00,2026-04-22 05:47:37.601075
4,12351.0,retail_customer_events,0,2182,2026-04-08 07:33:57.668,2026-04-08 07:33:57.702,customer_activity,12351,Unspecified,2010-11-29 15:23:00,2026-04-22 05:47:37.601075
5,12352.0,retail_customer_events,1,2050,2026-04-08 07:08:06.964,2026-04-08 07:08:06.975,customer_activity,12352,Norway,2010-11-12 10:20:00,2026-04-22 05:47:37.601075
6,12353.0,retail_customer_events,1,1963,2026-04-08 06:44:27.791,2026-04-08 06:44:27.809,customer_activity,12353,Bahrain,2010-10-27 12:44:00,2026-04-22 05:47:37.601075
7,12355.0,retail_customer_events,1,1328,2026-04-07 15:04:20.287,2026-04-07 15:04:20.301,customer_activity,12355,Bahrain,2010-05-21 11:59:00,2026-04-22 05:47:37.601075
8,12356.0,retail_customer_events,0,1916,2026-04-08 06:21:00.406,2026-04-08 06:21:00.431,customer_activity,12356,Portugal,2010-10-11 09:42:00,2026-04-22 05:47:37.601075
9,12357.0,retail_customer_events,0,2110,2026-04-08 07:12:49.403,2026-04-08 07:12:49.437,customer_activity,12357,Switzerland,2010-11-16 10:05:00,2026-04-22 05:47:37.601075


In [5]:
from pyspark.sql.types import *
customers_schema = StructType([
    StructField("event_type", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("country", StringType(), True),
    StructField("first_seen_time", StringType(), True)
])

In [7]:
from pyspark.sql.functions import *
customers_parsed = customers_df.withColumn(
    "parsed_json",
    from_json(col("raw_json"), customers_schema)
)

In [19]:
customers_parsed.limit(5).toPandas()

,kafka_key,raw_json,topic,partition,offset,kafka_timestamp,bronze_ingestion_time,silver_ingestion_time,parsed_json
0,16701.0,"{""event_type"": ""customer_activity"", ""customer_...",retail_customer_events,0,69,2026-04-07 12:25:08.065,2026-04-07 12:26:00.403,2026-04-09 06:02:56.171988,"(customer_activity, 16701, United Kingdom, None)"
1,18102.0,"{""event_type"": ""customer_activity"", ""customer_...",retail_customer_events,1,1,2026-04-07 12:23:04.558,2026-04-07 12:26:00.403,2026-04-09 06:02:56.171988,"(customer_activity, 18102, United Kingdom, None)"
2,16150.0,"{""event_type"": ""customer_activity"", ""customer_...",retail_customer_events,0,48,2026-04-07 12:24:19.053,2026-04-07 12:26:00.403,2026-04-09 06:02:56.171988,"(customer_activity, 16150, United Kingdom, None)"
3,15362.0,"{""event_type"": ""customer_activity"", ""customer_...",retail_customer_events,1,0,2026-04-07 12:23:03.522,2026-04-07 12:26:00.403,2026-04-09 06:02:56.171988,"(customer_activity, 15362, United Kingdom, None)"
4,15680.0,"{""event_type"": ""customer_activity"", ""customer_...",retail_customer_events,1,64,2026-04-07 12:25:53.883,2026-04-07 12:26:00.403,2026-04-09 06:02:56.171988,"(customer_activity, 15680, United Kingdom, None)"


In [9]:
import json
rows = customers_parsed.select("raw_json").limit(5).collect()

for r in rows:
    print(json.dumps(json.loads(r["raw_json"]), indent=2))

{
  "event_type": "customer_activity",
  "customer_id": 16701,
  "country": "United Kingdom",
  "first_seen_time": "2009-12-01 19:19:00",
  "ingestion_time": "2026-04-07 12:25:08.065857"
}
{
  "event_type": "customer_activity",
  "customer_id": 18102,
  "country": "United Kingdom",
  "first_seen_time": "2009-12-01 09:24:00",
  "ingestion_time": "2026-04-07 12:23:04.557788"
}
{
  "event_type": "customer_activity",
  "customer_id": 16150,
  "country": "United Kingdom",
  "first_seen_time": "2009-12-01 14:02:00",
  "ingestion_time": "2026-04-07 12:24:19.053915"
}
{
  "event_type": "customer_activity",
  "customer_id": 15362,
  "country": "United Kingdom",
  "first_seen_time": "2009-12-01 09:08:00",
  "ingestion_time": "2026-04-07 12:23:03.522096"
}
{
  "event_type": "customer_activity",
  "customer_id": 15680,
  "country": "United Kingdom",
  "first_seen_time": "2009-12-02 12:15:00",
  "ingestion_time": "2026-04-07 12:25:53.883067"
}


In [10]:
customers_flat = customers_parsed.select(
    "kafka_key",
    "topic",
    "partition",
    "offset",
    "kafka_timestamp",
    "bronze_ingestion_time",
    "silver_ingestion_time",
    col("parsed_json.*")
)

customers_clean = customers_flat.withColumn(
    "first_seen_time",
    to_timestamp(col("first_seen_time"), "yyyy-MM-dd HH:mm:ss")
)

In [12]:
customers_clean.filter(col("first_seen_time").isNull()).count()

0

In [13]:
customers_clean.count()

4382

In [17]:
spark.catalog.clearCache()